## Let's simulate OptimAc on the data collected by the survey

### Data import and formating into participant lists

In [1]:
import pandas as pan
from participant_real import Participant

dfs = [f0,f1]  = [pan.read_csv("../data/experimental2/filtered/f0.csv"), pan.read_csv("../data/experimental2/filtered/f1.csv")]
f1

,n,action_name,motivation,feedback
0,1,weekly,2,Oui
1,1,monthly,2,Oui
2,1,notatall,-1,Oui
3,2,weekly,1,"Oui, mais pas sûr"
4,2,monthly,2,Oui
...,...,...,...,...
208,126,monthly,-2,Non
209,126,notatall,-1,Non
210,128,weekly,2,Oui
211,128,monthly,2,Oui


In [2]:
action_names = [["weekly","monthly","as_usual"],["weekly","monthly","notatall"]]
participants = [[],[]]

extract_data = lambda df :list(df.groupby("n")["motivation"].apply(list).items())
# data is extracted from the dataframe as a list of (id, motivations list)
id_and_motivation_lists = [extract_data(f0), extract_data(f1)]

for scenario in range (2):
    for (id_number,motivationlist) in id_and_motivation_lists[scenario]:
        motiv_dict = {k:v for (k,v) in zip(action_names[scenario],motivationlist)}
        participants[scenario].append(Participant(id_number,motiv_dict).to_dict())

part_dfs = [pan.DataFrame(participants[scenario]) for scenario in range(2)]


## Simulation

We use target distribution $d_i = 1/3$ with $pa_i = 10$

In [3]:
from Optimac import allocate_actions_to_participants

d = 1/3
pa = 10

simulate_scenario = lambda scenario : allocate_actions_to_participants(part_dfs[scenario],{ac : {"target": d, "current":0,"minimum" :pa}for ac in action_names[scenario]})

results = [simulate_scenario(i) for i in range(2)]

## Results analysis

In [ ]:
import matplotlib.pyplot as plt 
import random as rd
from time import time

rd.seed(time())
res_dfs = [pan.DataFrame(r,columns=["id","action","motivaion"]) for r in results]

print(res_dfs[0].groupby("action")["action"].count())
print(res_dfs[1].groupby("action")["action"].count())

feedback_map = {"Oui":1, "Oui, mais pas sûr":0.5,"Non":0}

remaining =[]

for scenario in range(2):
    feedbacks = dfs[scenario].merge(
        res_dfs[scenario],
        left_on=["n", "action_name"],
        right_on=["id", "action"],
        how="inner"
    )[["id","action","motivaion","feedback"]] 
    print(feedbacks["feedback"].unique())  
    feedbacks["feedback"].map(feedback_map) 
    random_dropouts = [rd.random() for _ in range(len(feedbacks))]
    remaining_here = feedbacks[[feedbacks["feedback"]]]
    remaining.append()

action
as_usual    25
monthly     24
weekly      24
Name: action, dtype: int64
action
monthly     23
notatall    24
weekly      24
Name: action, dtype: int64
['Oui' 'Non' 'Oui, mais pas sûr']


TypeError: Series.map() got an unexpected keyword argument 'axis'